# Exploratory Data Analysis: Cardiovascular Health Dataset
### Project: Cardiovascular Health Monitoring and Prediction Using Artificial Neural Networks
This notebook analyzes the 70,000-record Kaggle Cardiovascular Disease dataset to investigate feature distributions, physiological anomalies, class balance, and correlations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading & Inspection

In [2]:
# Load dataset using semicolon delimiter
df = pd.read_csv('../data/raw/cardio_train.csv', sep=';')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Feature Engineering & Conversion
- Convert `age` from days to years (`age / 365.25`)
- Compute Body Mass Index (BMI = $\frac{\text{weight}}{\text{height}^2}$)

In [3]:
df['age_years'] = (df['age'] / 365.25).round(1)
df['bmi'] = (df['weight'] / ((df['height'] / 100.0) ** 2)).round(2)
df[['age', 'age_years', 'height', 'weight', 'bmi', 'cardio']].describe()

## 3. Detecting Physiological Outliers & Cleaning

In [4]:
# Blood pressure physiological plausibility filter
valid_bp = (df['ap_hi'] >= 70) & (df['ap_hi'] <= 240) & (df['ap_lo'] >= 40) & (df['ap_lo'] <= 160) & (df['ap_hi'] > df['ap_lo'])
valid_phys = (df['height'] >= 125) & (df['height'] <= 220) & (df['weight'] >= 35) & (df['weight'] <= 200)

clean_df = df[valid_bp & valid_phys].copy()
print(f"Removed {len(df) - len(clean_df)} implausible records.")
print(f"Retained {len(clean_df)} records ({len(clean_df)/len(df)*100:.2f}%).")

## 4. Target Class Balance Analysis

In [5]:
target_counts = clean_df['cardio'].value_counts()
print("Target counts:\n", target_counts)
clean_df['cardio'].value_counts(normalize=True).plot(kind='bar', color=['#3b82f6', '#ef4444'])
plt.title('Target Distribution (cardio: 0 vs 1)')
plt.xlabel('Cardiovascular Disease Class')
plt.ylabel('Proportion')
plt.show()

## 5. Correlation Heatmap

In [6]:
features = ['age_years', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']
corr = clean_df[features].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Pearson Correlation Heatmap')
plt.show()